In [1]:
# Pin here for fix of CPU offloading bug; implemented in docker /uv
#%pip install "transformers==4.57.3" "accelerate==1.12.0" "bitsandbytes==0.49.1" "vllm<0.22.0"

In [2]:
from experiments_vllm import *
from data import *
from config_subject_skew import *

/home/dylan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
# Environment fixes
import os
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"

In [4]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [5]:
# Get data: generalize prompts and per-subject
print(subjects)
dataset_general = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)
datasets_subjects = [get_data_mmlu(n_samples=n_samples, shuffle_seed=seed, subset=s) for s in subjects]
prompts_general = format_prompts_mmlu(dataset_general)
prompts_subjects = [format_prompts_mmlu(d) for d in datasets_subjects]

['abstract_algebra']
Streaming cais/mmlu (all) (samples: 100)...
Streaming cais/mmlu (abstract_algebra) (samples: 100)...


In [6]:
# Get model
model, tokenizer = load_model(model_id, enable_bnb=True)
probe = MoEProbeMistral(model)

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

load_tokenizer: no pad token defined for mistralai/Mixtral-8x7B-Instruct-v0.1, using eos_token ('</s>') as pad_token.
MoEHook: Scanning model for routers...
MoEHook: Attached probes to 32 router layers.
MoEHook: Model has 32 routers each with 8 experts and selects k=2 at each layer.


In [7]:
# Make trace directories
if not os.path.isdir(trace_path_general):
    os.mkdir(trace_path_general)
for p in trace_paths_subjects:
    if not os.path.isdir(p):
        os.mkdir(p)

In [8]:
# Measure throughput and record traces over generalized prompts
overall_results = await measure_vllm_throughput(model_id,
                                                prompts_general,
                                                seed=seed,
                                                max_new_tokens=max_new_tokens,
                                                max_model_len=max_model_len,
                                                batch_size=batch_size,
                                                concurrency_limit=batch_size*4,
                                                gpu_memory_utilization=gpu_memory_utilization,
                                                n_gpus=n_gpus,
                                                n_warmup_samples=n_warmup_samples,
                                                print_output=False,
                                                enable_expert_parallel=enable_expert_parallel,
                                                enable_prefix_caching=enable_prefix_caching,
                                                trace_dir=trace_path_general)

Starting vLLM server for mistralai/Mixtral-8x7B-Instruct-v0.1...
Waiting for server to initialize ...
WARNING 08-13 13:02:15 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
WARNING 08-13 13:02:18 [profiler.py:129] Using 'torch' profiler with delay_iterations or max_iterations while ignore_frontend is False may result in high overhead.
(APIServer pid=1588447) INFO 08-13 13:02:18 [utils.py:306] 
(APIServer pid=1588447) INFO 08-13 13:02:18 [utils.py:306]        █     █     █▄   ▄█
(APIServer pid=1588447) INFO 08-13 13:02:18 [utils.py:306]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.21.0
(APIServer pid=1588447) INFO 08-13 13:02:18 [utils.py:306]   █▄█▀ █     █     █     █  model   mistralai/Mixtral-8x7B-Instruct-v0.1
(APIServer pid=1588447) INFO 08-13 13:02:18 [utils.py:306]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer

[rank0]:[W813 13:02:43.522856697 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


(EngineCore pid=1588515) ERROR 08-13 13:02:45 [core.py:1140] EngineCore failed to start.
(EngineCore pid=1588515) ERROR 08-13 13:02:45 [core.py:1140] Traceback (most recent call last):
(EngineCore pid=1588515) ERROR 08-13 13:02:45 [core.py:1140]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1114, in run_engine_core
(EngineCore pid=1588515) ERROR 08-13 13:02:45 [core.py:1140]     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=1588515) ERROR 08-13 13:02:45 [core.py:1140]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=1588515) ERROR 08-13 13:02:45 [core.py:1140]     return func(*args, **kwargs)
(EngineCore pid=1588515) ERROR 08-13 13:02:45 [core.py:1140]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 880, in __init__
(EngineCore pid=1588515) ERROR 08-13 13:02:45 [core.py:1140]     super().__init__(
(En

(EngineCore pid=1588515) Process EngineCore:
(EngineCore pid=1588515) Traceback (most recent call last):
(EngineCore pid=1588515)   File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=1588515)     self.run()
(EngineCore pid=1588515)   File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore pid=1588515)     self._target(*self._args, **self._kwargs)
(EngineCore pid=1588515)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1144, in run_engine_core
(EngineCore pid=1588515)     raise e
(EngineCore pid=1588515)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1114, in run_engine_core
(EngineCore pid=1588515)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=1588515)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=1588515)     return func(*a

An error occurred during inference: vLLM server process terminated unexpectedly.


In [9]:
# Measure throughput and record traces over single MMLU subject
subject_id = 0
subject_results = await measure_vllm_throughput(model_id,
                                                prompts_subjects[subject_id],
                                                seed=seed,
                                                max_new_tokens=max_new_tokens,
                                                max_model_len=max_model_len,
                                                batch_size=batch_size,
                                                concurrency_limit=batch_size*4,
                                                gpu_memory_utilization=gpu_memory_utilization,
                                                n_gpus=n_gpus,
                                                n_warmup_samples=n_warmup_samples,
                                                print_output=False,
                                                enable_expert_parallel=enable_expert_parallel,
                                                enable_prefix_caching=enable_prefix_caching,
                                                trace_dir=trace_paths_subjects[subject_id])

Starting vLLM server for mistralai/Mixtral-8x7B-Instruct-v0.1...
Waiting for server to initialize ...
WARNING 08-13 13:02:57 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
WARNING 08-13 13:02:59 [profiler.py:129] Using 'torch' profiler with delay_iterations or max_iterations while ignore_frontend is False may result in high overhead.
(APIServer pid=1588694) INFO 08-13 13:02:59 [utils.py:306] 
(APIServer pid=1588694) INFO 08-13 13:02:59 [utils.py:306]        █     █     █▄   ▄█
(APIServer pid=1588694) INFO 08-13 13:02:59 [utils.py:306]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.21.0
(APIServer pid=1588694) INFO 08-13 13:02:59 [utils.py:306]   █▄█▀ █     █     █     █  model   mistralai/Mixtral-8x7B-Instruct-v0.1
(APIServer pid=1588694) INFO 08-13 13:02:59 [utils.py:306]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer

[rank0]:[W813 13:03:24.256105231 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


(EngineCore pid=1588756) ERROR 08-13 13:03:25 [core.py:1140] EngineCore failed to start.
(EngineCore pid=1588756) ERROR 08-13 13:03:25 [core.py:1140] Traceback (most recent call last):
(EngineCore pid=1588756) ERROR 08-13 13:03:25 [core.py:1140]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1114, in run_engine_core
(EngineCore pid=1588756) ERROR 08-13 13:03:25 [core.py:1140]     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=1588756) ERROR 08-13 13:03:25 [core.py:1140]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=1588756) ERROR 08-13 13:03:25 [core.py:1140]     return func(*args, **kwargs)
(EngineCore pid=1588756) ERROR 08-13 13:03:25 [core.py:1140]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 880, in __init__
(EngineCore pid=1588756) ERROR 08-13 13:03:25 [core.py:1140]     super().__init__(
(En

(EngineCore pid=1588756) Process EngineCore:
(EngineCore pid=1588756) Traceback (most recent call last):
(EngineCore pid=1588756)   File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=1588756)     self.run()
(EngineCore pid=1588756)   File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore pid=1588756)     self._target(*self._args, **self._kwargs)
(EngineCore pid=1588756)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1144, in run_engine_core
(EngineCore pid=1588756)     raise e
(EngineCore pid=1588756)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1114, in run_engine_core
(EngineCore pid=1588756)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=1588756)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=1588756)     return func(*a

An error occurred during inference: vLLM server process terminated unexpectedly.


In [10]:
# Save results
#import pickle
#with open(results_file_balanced, 'wb') as file:
#    pickle.dump(balanced_results, file)
#with open(results_file_imbalanced, 'wb') as file:
#    pickle.dump(imbalanced_results, file)